# for testing, Gemma 4 Developer Agent

## 1. Environment Configuration and Package Installation

In [1]:
import glob
import importlib
import logging
import os
import shutil
import subprocess
import sys
from pathlib import Path

# Configure environment variables for offline vLLM serving and LiteLLM routing
os.environ['TRANSFORMERS_NO_TF'] = '1'
os.environ['TRANSFORMERS_NO_FLAX'] = '1'
os.environ['TRANSFORMERS_OFFLINE'] = '1'
os.environ['VLLM_NO_USAGE_STATS'] = '1'
os.environ['VLLM_WORKER_MULTIPROC_METHOD'] = 'spawn'
os.environ['VLLM_ENGINE_READY_TIMEOUT_S'] = '1200'
os.environ['VLLM_MEMORY_PROFILER_ESTIMATE_CUDAGRAPHS'] = '1'
os.environ['NCCL_DEBUG'] = 'WARN'
os.environ['LITELLM_LOCAL_MODEL_COST_MAP'] = 'True'
os.environ['LITELLM_TELEMETRY'] = 'False'
os.environ['LITELLM_LOG'] = 'WARNING'
os.environ['OPENAI_API_KEY'] = os.environ.get('OPENAI_API_KEY', 'EMPTY')
os.environ['LOCAL_API_KEY'] = os.environ.get('LOCAL_API_KEY', 'EMPTY')
os.environ['PYTORCH_CUDA_ALLOC_CONF'] = 'expandable_segments:True'
os.environ['OTEL_SDK_DISABLED'] = 'true'
os.environ['OTEL_PYTHON_DISABLED'] = 'true'


def configure_logging() -> None:
    """Silence verbose third-party loggers."""
    for name in (
        'LiteLLM',
        'LiteLLM Router',
        'LiteLLM Proxy',
        'litellm',
        'httpx',
        'httpcore',
        'openai',
        'urllib3',
        'opentelemetry',
        'opentelemetry.context',
    ):
        lg = logging.getLogger(name)
        lg.setLevel(logging.WARNING)
        if 'litellm' in name.lower():
            lg.propagate = False
            lg.handlers.clear()


configure_logging()

WHEELHOUSE_DIR = Path('/kaggle/input/datasets/metric/gemma-4-developer-agent-wheelhouse')

# Remove broken cutlass .pth hooks if present
for pth_pattern in (
    '/usr/local/lib/python*/dist-packages/*cutlass*.pth',
    '/usr/local/lib/python*/site-packages/*cutlass*.pth',
):
    for pth in glob.glob(pth_pattern):
        try:
            os.unlink(pth)
        except OSError:
            pass

# Restore PEP 440 '+cu128' wheel filenames stripped by Kaggle dataset uploads
tmp_whl = Path('/tmp/wheelhouse')
tmp_whl.mkdir(parents=True, exist_ok=True)
for w in WHEELHOUSE_DIR.glob('*.whl'):
    if 'cutlass' in w.name.lower():
        continue
    target_name = (
        w.name.replace('cu128', '+cu128')
        if ('cu128' in w.name and '+' not in w.name)
        else w.name
    )
    target = tmp_whl / target_name
    if not target.exists():
        try:
            os.symlink(w, target)
        except OSError:
            shutil.copyfile(w, target)

wheels = sorted(str(w) for w in tmp_whl.glob('*.whl'))
print(f'Installing {len(wheels)} wheels from {WHEELHOUSE_DIR}...')
subprocess.run(
    [sys.executable, '-m', 'pip', 'install', '-q', '--no-deps', '--force-reinstall', *wheels],
    check=True,
)
importlib.invalidate_caches()
configure_logging()
print('Wheelhouse installation complete.')

Installing 41 wheels from /kaggle/input/datasets/metric/gemma-4-developer-agent-wheelhouse...
Wheelhouse installation complete.


## 2. Competition Dataset and Sample Submission,
## define model name, load tasks

In [2]:
# read model name from submission_x.zip
import yaml

submission_x = 'submission-6-zip'
Submission_DIR = Path(f'/kaggle/input/datasets/prince22466/{submission_x}')

config_model = yaml.load(
    (Submission_DIR / "agent.yaml").read_text(encoding="utf-8"),
    Loader=yaml.BaseLoader,
)

MODEL_NAME = config_model["model"]
print(MODEL_NAME)

gemma-4-e2b-it-qat-w4a16-ct


In [3]:
import re
from swegemma.models import load_tasks

DATA_DIR = Path('/kaggle/input/competitions/gemma-4-developer-agent')
WORKING_DIR = Path('/kaggle/working')
WORKING_DIR.mkdir(parents=True, exist_ok=True)


# Copy sample_submission to a writable directory for customization and packaging
SAMPLE_SUBMISSION_SRC = DATA_DIR / 'sample_submission'
Submission_DIR
AGENT_DIR = WORKING_DIR / 'sample_submission'
if AGENT_DIR.exists():
    shutil.rmtree(AGENT_DIR)
shutil.copytree(Submission_DIR, AGENT_DIR)



# Load tasks from the published dataset
TASKS_PATH = DATA_DIR / 'tasks.jsonl'
tasks = load_tasks(TASKS_PATH)

print(f'Data directory: {DATA_DIR}')
print(f'Agent directory: {AGENT_DIR} (model={MODEL_NAME})')
print(f'Loaded {len(tasks)} tasks from {TASKS_PATH.name}')
for t in tasks[:5]:
    print(f'  - {t.instance_id} ({t.repo} @ {t.base_commit[:8]})')

Data directory: /kaggle/input/competitions/gemma-4-developer-agent
Agent directory: /kaggle/working/sample_submission (model=gemma-4-e2b-it-qat-w4a16-ct)
Loaded 129 tasks from tasks.jsonl
  - fastapi_15661 (fastapi/fastapi @ ee22a4b8)
  - fastapi_15588 (fastapi/fastapi @ cb83b83d)
  - fastapi_15589 (fastapi/fastapi @ 22b02e26)
  - fastapi_15030 (fastapi/fastapi @ 48d58ae3)
  - fastapi_14962 (fastapi/fastapi @ 1e78a36b)


## 3. Code Graph Functions

In [4]:
from swegemma import graph as sg

sample_task = tasks[0]
GRAPH_DIR = str(DATA_DIR / 'graphs')
EMBEDDINGS_DIR = str(DATA_DIR / 'embeddings')

# Load the pre-computed repository code graph and node embeddings for the sample task
repo_graph = sg.get_graph(
    repo_name=sample_task.repo,
    graph_dir=GRAPH_DIR,
    embeddings_dir=EMBEDDINGS_DIR,
    base_commit=sample_task.base_commit,
)
sample_nodes = list(repo_graph.nodes())
print(f'Graph for {sample_task.repo} ({sample_task.instance_id}):')
print(f'  Nodes: {repo_graph.number_of_nodes()}, Edges: {repo_graph.number_of_edges()}')

# Select a connected symbol and query structural neighbors (callers, callees, definitions)
query_node = next((n for n, deg in repo_graph.degree() if deg >= 5), sample_nodes[0])
neighbors = sg.get_neighbor(
    node=query_node,
    graph=repo_graph,
    max_neighbors=10,
)
print(f'\nNeighbors of {query_node!r} (showing up to 5):')
for n in neighbors[:5]:
    print(f'  - {n}')

# Search for semantically similar code nodes using pre-computed vector embeddings
similar = sg.get_similar_nodes(
    node=query_node,
    repo_name=sample_task.repo,
    k=5,
    graph=repo_graph,
    graph_dir=GRAPH_DIR,
    embeddings_dir=EMBEDDINGS_DIR,
    base_commit=sample_task.base_commit,
)
print(f'\nTop similar nodes to {query_node!r}:')
for item in similar:
    print(f"  - {item['node_name']} (similarity={item['similarity']:.4f})")

# Extract an induced subgraph connecting the query symbol and its neighbors
focal_nodes = [query_node, *neighbors[:4]]
subgraph = sg.get_induced_subgraph(repo_graph, focal_nodes)
print(
    f'\nInduced subgraph over {len(focal_nodes)} focal nodes: '
    f'{subgraph.number_of_nodes()} nodes, {subgraph.number_of_edges()} edges'
)

Graph for fastapi/fastapi (fastapi_15661):
  Nodes: 4263, Edges: 2430

Neighbors of 'custom_request_and_route.tutorial002_py310.ValidationErrorLoggingRoute.get_route_handler' (showing up to 5):
  - fastapi/fastapi;custom_request_and_route.tutorial002_py310.ValidationErrorLoggingRoute
  - fastapi/fastapi;fastapi.exceptions.HTTPException
  - fastapi/fastapi;fastapi.exceptions.RequestValidationError

Top similar nodes to 'custom_request_and_route.tutorial002_py310.ValidationErrorLoggingRoute.get_route_handler':
  - custom_request_and_route.tutorial002_an_py310.ValidationErrorLoggingRoute.get_route_handler (similarity=1.0000)
  - fastapi.routing.APIRouter.__init__ (similarity=0.9996)
  - fastapi.applications.FastAPI.setup (similarity=0.9993)
  - custom_request_and_route.tutorial002_an_py310.ValidationErrorLoggingRoute (similarity=0.9991)
  - custom_request_and_route.tutorial002_py310.ValidationErrorLoggingRoute (similarity=0.9991)

Induced subgraph over 4 focal nodes: 4 nodes, 6 edges


## 4. Start vLLM Server

In [5]:
import litellm
import torch
from adk_submission import VllmConfig, VllmServer, discover_adapters
from adk_submission.discovery import discover_declared_models
from swegemma.config import ALLOWED_ADAPTER_EXTENSIONS

litellm.drop_params = True

ALLOWED_MODEL_NAMES = frozenset({
    'gemma-4-31b-it-qat-w4a16-ct',
    'gemma-4-12b-it-qat-w4a16-ct',
    'gemma-4-e4b-it-qat-w4a16-ct',
    'gemma-4-e2b-it-qat-w4a16-ct',
})
MODEL_VLLM_EXTRA_ARGS = {
    'gemma-4-12b-it-qat-w4a16-ct': (
        '--hf-overrides',
        '{"architectures": ["Gemma4ForCausalLM"], "model_type": "gemma4"}',
    ),
}
_SAFE_PATH_COMPONENT = re.compile(r'[A-Za-z0-9][A-Za-z0-9._-]*')
INFERENCE_API_KEY = os.environ.get('LOCAL_API_KEY', 'EMPTY')

# Validate single canonical base model and discover any PEFT LoRA adapters in the submission directory.
# Note: sample_submission includes dummy LoRA adapters (adapters/main_lora and adapters/tool_lora)
# shaped for gemma-4-31b-it-qat-w4a16-ct; Cell 2 removes them automatically when MODEL_NAME != 31B.
declared_models = discover_declared_models(AGENT_DIR)
assert len(declared_models) == 1, f'Expected exactly 1 declared model, got: {sorted(declared_models)}'
declared_model = next(iter(declared_models))
assert declared_model in ALLOWED_MODEL_NAMES, f'Unsupported model {declared_model!r}. Allowed: {sorted(ALLOWED_MODEL_NAMES)}'

MODEL_PATH = Path(f'/kaggle/input/models/google/gemma-4/other/{declared_model}/2')
assert MODEL_PATH.exists(), f'Model weights not found at {MODEL_PATH}. Ensure the model input is attached.'

adapters = discover_adapters(str(AGENT_DIR), adapter_extensions=ALLOWED_ADAPTER_EXTENSIONS)
adapter_map = getattr(adapters, 'adapters', None) or {}
for adapter_name in adapter_map:
    assert _SAFE_PATH_COMPONENT.fullmatch(adapter_name), (
        f'Unsafe adapter name {adapter_name!r}: must match [A-Za-z0-9][A-Za-z0-9._-]*'
    )
detected_ranks = [
    r for info in adapter_map.values() if (r := getattr(info, 'rank', None)) is not None and r > 0
]

gpu_count = torch.cuda.device_count() if torch.cuda.is_available() else 1
tp_size = 4 if gpu_count >= 4 else (2 if gpu_count >= 2 else 1)

if 'server_instance' in globals() and server_instance is not None:
    server_instance.stop()

vllm_cfg = VllmConfig(
    model=str(MODEL_PATH),
    port=8000,
    host='127.0.0.1',
    tool_call_parser='gemma4',
    reasoning_parser='gemma4',
    max_model_len=32768,
    dtype='bfloat16' if (torch.cuda.is_available() and torch.cuda.is_bf16_supported()) else 'auto',
    gpu_memory_utilization=0.90,
    enable_auto_tool_choice=True,
    enable_lora=bool(adapter_map),
    max_loras=max(1, len(adapter_map)),
    max_lora_rank=max(detected_ranks) if detected_ranks else 128,
    scheduler_reserve_full_isl=False,
    tensor_parallel_size=tp_size,
    startup_timeout=60 * 20,
    extra_args=MODEL_VLLM_EXTRA_ARGS.get(declared_model, ()),
)
server_instance = VllmServer(vllm_cfg, adapter_manifest=adapters)
server_instance.start()
print(f'vLLM server started on {server_instance.base_url} (tp={tp_size})')

# Register model and LoRA adapter aliases for Google ADK
models = server_instance.create_model_registry(
    aliases=[declared_model],
    model_prefix='openai/',
    api_key=INFERENCE_API_KEY,
)

vLLM server started on http://127.0.0.1:8000/v1 (tp=4)


In [6]:
# print out vllm config
print(vllm_cfg)

VllmConfig(model='/kaggle/input/models/google/gemma-4/other/gemma-4-e2b-it-qat-w4a16-ct/2', port=8000, host='127.0.0.1', tool_call_parser='gemma4', reasoning_parser='gemma4', reasoning_config=None, default_chat_template_kwargs=None, limit_mm_per_prompt=None, speculative_config=None, kv_cache_dtype=None, max_model_len=32768, dtype='bfloat16', gpu_memory_utilization=0.9, enable_auto_tool_choice=True, enable_lora=False, max_loras=1, max_lora_rank=128, scheduler_reserve_full_isl=False, extra_args=(), startup_timeout=1200, tensor_parallel_size=4, health_check_interval=1.0, nvidia_lib_dir='/usr/local/nvidia/lib64')


## 5. Run  Inference and  Verification

In [7]:
from __future__ import annotations

# Evaluation controls: edit this block before running.
INHERIT = 'submission'  # Read this limit from eval_config.yaml, then harness defaults.
EVAL_OVERRIDES = {
    'max_time_minutes': None,           # None disables the per-task session timer.
    'max_tool_calls': INHERIT,           # Positive integer; None removes this budget.
    'max_turns': INHERIT,                # Positive integer; None uses the harness's 500-turn fallback.
    'max_total_tokens': INHERIT,         # Total task tokens; None removes this budget.
    'max_cost_usd': INHERIT,             # Provider-reported USD; None removes this budget.
    'timeout_seconds': INHERIT,          # Single command timeout, separate from task time.
    'verification_timeout_seconds': INHERIT,  # Final verification timeout.
    'max_stdout_chars': INHERIT,         # Characters returned per command.
    'max_file_lines': INHERIT,           # Lines returned per read_file call.
    'max_file_chars': INHERIT,           # Characters returned per read_file call.
}
TASK_IDS = []  # Explicit IDs take precedence over TASK_LIMIT.
TASK_LIMIT = 30  # None selects all tasks.
TASK_TYPES = {}  # Example: {'fastapi_15661': 'release_automation'}.
LIVE_AGENT_OBSERVER = True
CACHE_SETTINGS = {'min_tokens': 2048, 'ttl_seconds': 1800, 'cache_intervals': 10}
COMPACTION_SETTINGS = {'compaction_interval': 15, 'overlap_size': 2,
                       'token_threshold': 14336, 'event_retention_size': 5}
# Per-response output/thinking token limits come from the agent's sampling YAML.
# Model context length comes from the earlier vLLM setup (max_model_len).

# Diagnostic implementation follows.
"""Notebook-only observers and reports. Never included in submission.zip.

Provider usage is counted once, using model observations OR the ATIF fallback.
Stage labels are inferred. Parallel/nested wall times must not be added to task
duration. Observers pass through original generators and retire task contexts.
"""


import asyncio
import concurrent.futures
import contextlib
import contextvars
import csv
import dataclasses
import hashlib
import importlib.metadata
import inspect
import io
import json
import logging
import math
import re
import shutil
import statistics
import threading
import time
import traceback
import zipfile
from collections import Counter, defaultdict
from datetime import datetime, timezone
from pathlib import Path

DIAGNOSTICS_VERSION = '2.2-result-serialization'


def value(obj, key, default=None):
    return obj.get(key, default) if isinstance(obj, dict) else getattr(obj, key, default)


def number(obj):
    return obj if isinstance(obj, (int, float)) and not isinstance(obj, bool) and math.isfinite(obj) else None


def plain(obj, _seen=None):
    """Serialize results, without dumping arbitrary object reprs or credentials."""
    if obj is None or isinstance(obj, (str, bool, int)):
        return obj
    if isinstance(obj, float):
        return obj if math.isfinite(obj) else None
    if isinstance(obj, (Path, datetime)):
        return str(obj)
    seen = set() if _seen is None else _seen
    if id(obj) in seen:
        return {'unserialized_type': type(obj).__name__, 'reason': 'cyclic_reference'}
    seen.add(id(obj))
    try:
        if isinstance(obj, dict):
            return {str(k): '[redacted]' if re.search(r'api.?key|authorization|password|secret', str(k), re.I)
                    else plain(v, seen) for k, v in obj.items()}
        if isinstance(obj, (list, tuple, set)):
            return [plain(v, seen) for v in obj]
        if hasattr(obj, 'model_dump'):
            # JSON mode rejects arbitrary runtime types such as SessionTrace.
            # Python mode keeps them available for safe recursive conversion.
            try:
                dumped = obj.model_dump(mode='python')
            except Exception:
                dumped = vars(obj) if hasattr(obj, '__dict__') else {'unserialized_type': type(obj).__name__}
            return plain(dumped, seen)
        if dataclasses.is_dataclass(obj) and not isinstance(obj, type):
            # asdict() deep-copies opaque objects and cannot handle cycles.
            return plain({field.name: getattr(obj, field.name) for field in dataclasses.fields(obj)}, seen)
        if hasattr(obj, '__dict__'):
            return plain(vars(obj), seen)
        return {'unserialized_type': type(obj).__name__}
    finally:
        seen.remove(id(obj))


def write_json(path, obj):
    Path(path).write_text(json.dumps(plain(obj), ensure_ascii=False, indent=2) + '\n', encoding='utf-8')


def append_json(path, obj):
    with Path(path).open('a', encoding='utf-8') as handle:
        handle.write(json.dumps(plain(obj), ensure_ascii=False) + '\n')


def write_csv(path, rows, fields=None):
    rows = list(rows)
    fields = fields or sorted({key for row in rows for key in row}) or ['no_records']
    with Path(path).open('w', encoding='utf-8', newline='') as handle:
        writer = csv.DictWriter(handle, fieldnames=fields, extrasaction='ignore')
        writer.writeheader()
        for row in rows:
            writer.writerow({k: json.dumps(v, ensure_ascii=False) if isinstance(v, (dict, list)) else v
                             for k, v in row.items()})


def usage_metrics(usage):
    def pick(*keys):
        return next((n for key in keys if (n := number(value(usage, key))) is not None), None)
    return {
        'input_tokens': pick('prompt_tokens', 'input_tokens', 'prompt_token_count'),
        'output_tokens': pick('completion_tokens', 'output_tokens', 'candidates_token_count'),
        'total_tokens': pick('total_tokens', 'total_token_count'),
        'cached_tokens': pick('cached_tokens', 'cached_content_token_count'),
        'reasoning_tokens': pick('reasoning_tokens', 'thoughts_token_count'),
    }


def stage_for(agent, calls, helpers=()):
    """One stage per request; tool-call payloads are part of its output usage."""
    agent = (agent or '').lower()
    if 'verif' in agent:
        return 'verification'
    if 'review' in agent:
        return 'review'
    if 'analy' in agent or 'locator' in agent:
        return 'localization'
    stages = set()
    for call in calls:
        name = value(call, 'function_name', value(call, 'name', ''))
        args = value(call, 'arguments', value(call, 'args', {})) or {}
        if name in helpers:
            stages.add('delegation')
        elif name in {'edit_file', 'write_file'}:
            stages.add('implementation')
        elif name == 'submit_patch':
            stages.add('submission')
        elif name == 'run_command':
            command = str(value(args, 'command', ''))
            if re.search(r'\b(pytest|unittest|nosetests|tox)\b|\bassert\b', command):
                stages.add('verification')
            elif re.search(r'\b(grep|rg|find|ls|cat|head|sed)\b', command):
                stages.add('localization')
            else:
                stages.add('unknown')
        elif name in {'read_file', 'get_code_neighbors', 'search_similar_code', 'get_code_subgraph'}:
            stages.add('localization')
        else:
            stages.add('unknown')
    return next(iter(stages)) if len(stages) == 1 else 'mixed' if stages else 'unknown'


def event_parts(event):
    """Capture visible answers and tool exchanges, excluding thought text."""
    calls, responses, texts = [], [], []
    for part in value(value(event, 'content', {}), 'parts', []) or []:
        if value(part, 'thought', False):
            continue
        call, response = value(part, 'function_call'), value(part, 'function_response')
        if call is not None:
            calls.append({'tool_call_id': value(call, 'id'), 'function_name': value(call, 'name'),
                          'arguments': plain(value(call, 'args', {}))})
        if response is not None:
            responses.append({'tool_call_id': value(response, 'id'), 'function_name': value(response, 'name'),
                              'response': plain(value(response, 'response'))})
        if value(part, 'text'):
            texts.append(value(part, 'text'))
    return calls, responses, texts


def trace_clock(obj):
    extra = value(obj, 'extra', {}) or {}
    return next((n for v in (value(extra, 'elapsed_s'), value(obj, 'elapsed_seconds'),
                             value(obj, 'relative_seconds')) if (n := number(v)) is not None), None)


def decode_result(obj):
    if isinstance(obj, str):
        try:
            return json.loads(obj)
        except json.JSONDecodeError:
            return obj
    return obj


def describe_trace(trace, helpers=()):
    """Parse ATIF-v1.7, including its nested extra and observation fields."""
    requests, tools, timeline, warnings = [], [], [], []
    steps = value(trace, 'steps', [])
    if not isinstance(steps, list):
        return {'requests': [], 'tools': [], 'timeline': [], 'warnings': ['Unsupported trace steps'], 'final_metrics': None}
    results = {}
    for step in steps:
        observation = value(step, 'observation', {}) or {}
        for observed in value(observation, 'results', []) or []:
            call_id = value(observed, 'source_call_id', value(observed, 'tool_call_id'))
            if call_id is not None:
                results[str(call_id)] = (decode_result(value(observed, 'content', value(observed, 'result'))), trace_clock(observed))
        if value(step, 'source') == 'tool':
            call_id = value(step, 'tool_call_id', value(step, 'source_call_id'))
            if call_id is not None:
                results[str(call_id)] = (decode_result(value(step, 'content', value(step, 'message'))), trace_clock(step))
    for index, step in enumerate(steps):
        if not isinstance(step, dict):
            warnings.append('Non-mapping step skipped')
            continue
        extra = step.get('extra') or {}
        agent = extra.get('author', step.get('author')) or 'unknown'
        elapsed = trace_clock(step)
        calls = step.get('tool_calls') or []
        calls = [c for c in calls if isinstance(c, dict)]
        common = {'step_id': step.get('step_id', index), 'agent': agent,
                  'elapsed_seconds': elapsed, 'source': 'atif'}
        if step.get('source', step.get('role')) in {'agent', 'assistant'}:
            usage = next((step[k] for k in ('metrics', 'usage', 'usage_metadata', 'token_usage')
                          if isinstance(step.get(k), dict)), {})
            requests.append({**common, 'request_id': f'atif:{index}', 'model': step.get('model_name'),
                             **usage_metrics(usage), 'model_request_wall_seconds': None,
                             'stage': stage_for(agent, calls, helpers), 'stage_provenance': 'inferred_from_author_and_actions',
                             'llm_call_count_reported': step.get('llm_call_count')})
        observation = step.get('observation') or {}
        for call in calls:
            function = call.get('function') or {}
            name = call.get('function_name', call.get('name', function.get('name')))
            call_id = str(call.get('tool_call_id', call.get('id', '')))
            start = trace_clock(call)
            start = start if start is not None else elapsed
            result, end = results.get(call_id, (decode_result(call.get('result', call.get('output'))), None))
            matching = 'call_id' if call_id in results else 'inline_result'
            if result is None and len(calls) == 1 and 'content' in observation:
                result, end = decode_result(observation['content']), trace_clock(observation)
                matching = 'same_step_single_call'
            row = {**common, 'tool_call_id': call_id or None, 'tool_name': name,
                   'arguments': decode_result(call.get('arguments', function.get('arguments', {}))),
                   'result': result, 'result_attribution': matching if result is not None else None,
                   'start_elapsed_seconds': start, 'end_elapsed_seconds': end,
                   'tool_wall_seconds': end - start if start is not None and end is not None and end >= start else None,
                   'status': value(result, 'status'), 'is_delegation': name in helpers}
            tools.append(row)
            timeline.append({**common, 'kind': 'delegate' if name in helpers else 'tool_call',
                             'to': name, 'tool_call_id': call_id or None, 'arguments': row['arguments']})
            if result is not None:
                timeline.append({**common, 'elapsed_seconds': end, 'kind': 'helper_response' if name in helpers else 'tool_result',
                                 'from': name, 'to': agent, 'tool_call_id': call_id or None, 'result': result})
    if any(tool['result'] is None for tool in tools):
        warnings.append('Some tool results are missing or cannot be matched safely')
    authors = {row['agent'] for row in requests}
    for helper in helpers:
        if any(t['tool_name'] == helper for t in tools) and helper not in authors:
            warnings.append(f'{helper}: delegation recorded, internal helper requests absent from ATIF')
    clocks = [trace_clock(s) for s in steps if isinstance(s, dict)]
    clocks += [t['end_elapsed_seconds'] for t in tools]
    clocks = [c for c in clocks if c is not None]
    return {'requests': requests, 'tools': tools, 'timeline': timeline, 'warnings': warnings,
            'final_metrics': value(trace, 'final_metrics'),
            'last_recorded_elapsed_seconds': max(clocks) if clocks else None,
            'observed_trace_window_seconds': max(clocks) - min(clocks) if clocks else None}


class DiagnosticObserver:
    """Task-scoped ADK observation; no requests, budgets or events are changed.

    BaseAgent catches nested AgentTool agents. LiteLlm records each model request
    once, including requests not exported by the parent session's trace. Separate
    processes and classes overriding these entry points may escape observation.
    """

    def __init__(self, directory, helpers=(), agent_class=None, model_class=None, enabled=True, instance_id=None):
        self.directory = Path(directory)
        self.directory.mkdir(parents=True, exist_ok=True)
        self.helpers = set(helpers)
        self.agent_class, self.model_class = agent_class, model_class
        self.enabled = enabled
        self.instance_id = instance_id
        self.requests, self.events, self.warnings = [], [], []
        self.seen, self.pending = set(), defaultdict(set)
        self.lock = threading.RLock()
        self.started = time.perf_counter()
        self.hooks = []
        self.active_token = None
        self.state = None
        self.retained_runtime_wrappers = []
        self.closed = False
        self.counter = Counter()

    def identifier(self, kind):
        with self.lock:
            self.counter[kind] += 1
            return f'{kind}_{self.counter[kind]}'

    def emit(self, row):
        if self.closed:
            return
        with self.lock:
            event = {'instance_id': self.instance_id, 'observer_elapsed_seconds': time.perf_counter() - self.started, **plain(row)}
            self.events.append(event)
            append_json(self.directory / 'events.jsonl', event)

    def safe(self, fn, *args):
        try:
            return fn(*args)
        except Exception as error:
            self.warnings.append(f'Observer recording failed: {type(error).__name__}: {error}')
            return None

    def observe_event(self, event, frame):
        calls, responses, texts = event_parts(event)
        native_id = value(event, 'id')
        signature = json.dumps(plain([calls, responses, texts, value(event, 'partial')]), sort_keys=True)
        key = (native_id or id(event), signature)
        with self.lock:
            if key in self.seen:
                return
            self.seen.add(key)
        author = value(event, 'author') or frame['agent']
        self.emit({'kind': 'agent_event', 'agent': author, 'agent_run_id': frame['agent_run_id'],
                   'parent_agent_run_id': frame['parent_agent_run_id'], 'event_id': native_id,
                   'invocation_id': value(event, 'invocation_id'), 'branch': value(event, 'branch'),
                   'partial': value(event, 'partial'), 'calls': calls, 'responses': responses,
                   'visible_text': '\n'.join(texts) or None,
                   'error_code': value(event, 'error_code'), 'error_message': value(event, 'error_message')})
        for call in calls:
            if call['tool_call_id']:
                self.pending[(frame['agent_run_id'], call['function_name'])].add(call['tool_call_id'])
        for response in responses:
            if response['tool_call_id']:
                self.pending[(frame['agent_run_id'], response['function_name'])].discard(response['tool_call_id'])

    def __enter__(self):
        if not self.enabled:
            self.warnings.append('Live ADK observer disabled; using runtime trace only')
            return self
        try:
            if self.agent_class is None:
                from google.adk.agents.base_agent import BaseAgent
                self.agent_class = BaseAgent
            if self.model_class is None:
                from google.adk.models.lite_llm import LiteLlm
                self.model_class = LiteLlm
        except ImportError as error:
            self.warnings.append(f'Live ADK observer unavailable: {error}')
            return self
        # Keep the dispatcher state on the ADK class so it survives notebook
        # module reloads and runtime wrappers such as ModelRetryPlugin.
        for cls, method in [(self.agent_class, 'run_async'), (self.model_class, 'generate_content_async')]:
            original = getattr(cls, method, None)
            if getattr(original, '_diagnostic_hook', False) and not getattr(original, '_diagnostic_dispatch', False):
                raise RuntimeError('Restart the Kaggle kernel before using updated diagnostics: an older observer hook is still retained')
        state = getattr(self.agent_class, '_notebook_diagnostic_state', None)
        if state is None:
            state = {name: contextvars.ContextVar('diagnostic_' + name, default=default) for name, default in [
                ('active', None), ('frame', None), ('agent_guard', frozenset()), ('model_guard', frozenset())]}
            setattr(self.agent_class, '_notebook_diagnostic_state', state)
        self.state = state
        if state['active'].get() is not None:
            raise RuntimeError('An observer is already active in this task context')
        self.active_token = state['active'].set(self)
        for cls, method, kind in [(self.agent_class, 'run_async', 'agent'),
                                  (self.model_class, 'generate_content_async', 'model')]:
            original = getattr(cls, method, None)
            if original is None:
                self.warnings.append(f'Cannot install {kind} observer: missing method')
                continue
            if getattr(original, '_diagnostic_dispatch', None) is state and getattr(original, '_diagnostic_version', None) == DIAGNOSTICS_VERSION:
                self.hooks.append((cls, method, None, None))
                continue
            if kind == 'agent':
                async def wrapped_agent(agent, *args, _original=original, **kwargs):
                    observer = state['active'].get()
                    guards = state['agent_guard'].get()
                    if observer is None or observer.closed or id(agent) in guards:
                        async with contextlib.aclosing(_original(agent, *args, **kwargs)) as stream:
                            async for event in stream:
                                yield event
                        return
                    parent = state['frame'].get()
                    name = value(agent, 'name', 'unknown')
                    candidates = observer.pending.get((value(parent, 'agent_run_id'), name), set())
                    frame = {'agent': name, 'agent_run_id': observer.identifier('agent_run'),
                             'parent_agent_run_id': value(parent, 'agent_run_id'),
                             'parent_tool_call_id': next(iter(candidates)) if len(candidates) == 1 else None}
                    observer.safe(observer.emit, {'kind': 'agent_start', **frame})
                    started = time.perf_counter()
                    status = 'completed'
                    event_count = 0
                    stream = _original(agent, *args, **kwargs).__aiter__()
                    try:
                        while True:
                            # A consumer may close this generator in another Task.
                            # Set/reset context within each await, never across yield.
                            token = state['frame'].set(frame)
                            guard_token = state['agent_guard'].set(guards | {id(agent)})
                            try:
                                event = await anext(stream)
                            except StopAsyncIteration:
                                break
                            finally:
                                state['agent_guard'].reset(guard_token)
                                state['frame'].reset(token)
                            event_count += 1
                            observer.safe(observer.observe_event, event, frame)
                            yield event
                    except GeneratorExit:
                        status = 'response_received' if event_count else 'closed_without_response'
                        raise
                    except BaseException:
                        status = 'interrupted'
                        raise
                    finally:
                        token = state['frame'].set(frame)
                        guard_token = state['agent_guard'].set(guards | {id(agent)})
                        try:
                            await stream.aclose()
                        finally:
                            state['agent_guard'].reset(guard_token)
                            state['frame'].reset(token)
                            observer.safe(observer.emit, {'kind': 'agent_end', **frame, 'status': status,
                                                         'inclusive_wall_seconds': time.perf_counter() - started})
                wrapper = wrapped_agent
            else:
                async def wrapped_model(model, *args, _original=original, **kwargs):
                    observer = state['active'].get()
                    if observer is None or observer.closed or id(model) in state['model_guard'].get():
                        async with contextlib.aclosing(_original(model, *args, **kwargs)) as stream:
                            async for response in stream:
                                yield response
                        return
                    frame = state['frame'].get() or {'agent': 'unknown', 'agent_run_id': None}
                    request = args[0] if args else kwargs.get('llm_request')
                    request_config = value(request, 'config', {}) or {}
                    row = {'request_id': observer.identifier('request'), **frame,
                           'instance_id': observer.instance_id,
                           'model': value(request, 'model') or value(model, 'model'), 'source': 'live_adk',
                           'adk_request_config': {k: plain(value(request_config, k)) for k in
                               ('temperature', 'top_p', 'top_k', 'max_output_tokens', 'thinking_config',
                                'seed', 'presence_penalty', 'frequency_penalty')},
                           'start_observer_seconds': time.perf_counter() - observer.started,
                           'status': 'started', 'response_chunks': 0}
                    observer.safe(observer.emit, {'kind': 'model_start', **row})
                    started, last_usage, calls = time.perf_counter(), None, []
                    active_seconds, last_received = 0.0, None
                    try:
                        async with contextlib.aclosing(_original(model, *args, **kwargs)) as stream:
                            while True:
                                advance_started = time.perf_counter()
                                guard_token = state['model_guard'].set(state['model_guard'].get() | {id(model)})
                                try:
                                    response = await anext(stream)
                                except StopAsyncIteration:
                                    break
                                finally:
                                    active_seconds += time.perf_counter() - advance_started
                                    state['model_guard'].reset(guard_token)
                                last_received = time.perf_counter()
                                row['response_chunks'] += 1
                                row.update(last_response_partial=value(response, 'partial'),
                                           last_response_turn_complete=value(response, 'turn_complete'),
                                           finish_reason=plain(value(value(response, 'finish_reason'), 'value', value(response, 'finish_reason'))),
                                           model_error_code=plain(value(response, 'error_code')),
                                           model_error_message=value(response, 'error_message'))
                                usage = value(response, 'usage_metadata')
                                if usage is not None:
                                    last_usage = usage
                                response_calls, _, _ = event_parts(response)
                                calls.extend(response_calls)
                                yield response
                    except GeneratorExit:
                        row.update(status='response_received' if row['response_chunks'] else 'closed_without_response',
                                   close_reason='consumer_closed_generator')
                        raise
                    except BaseException as error:
                        row.update(status='interrupted', error_type=type(error).__name__)
                        raise
                    finally:
                        if row['status'] == 'started':
                            row['status'] = 'completed'
                        row.update(model_request_wall_seconds=last_received - started if last_received is not None else None,
                                   model_generator_active_seconds=active_seconds,
                                   end_observer_seconds=(last_received or time.perf_counter()) - observer.started,
                                   stage=stage_for(frame['agent'], calls, observer.helpers),
                                   stage_provenance='inferred_from_author_and_actions',
                                   **usage_metrics(last_usage))
                        if not observer.closed:
                            observer.requests.append(row)
                            observer.safe(append_json, observer.directory / 'requests.jsonl', row)
                            observer.safe(observer.emit, {'kind': 'model_end', **row})
                wrapper = wrapped_model
            wrapper._diagnostic_hook = True
            wrapper._diagnostic_dispatch = state
            wrapper._diagnostic_version = DIAGNOSTICS_VERSION
            setattr(cls, method, wrapper)
            self.hooks.append((cls, method, original, wrapper))
        return self

    def __exit__(self, *exc):
        # A third-party retry wrapper can retain our dispatcher. Retiring the
        # task context prevents it from holding onto that task's files/counters.
        self.closed = True
        if self.active_token is not None:
            self.state['active'].reset(self.active_token)
        for cls, name, original, wrapper in reversed(self.hooks):
            if wrapper is None:
                continue
            if getattr(cls, name) is wrapper:
                setattr(cls, name, original)
            else:
                self.retained_runtime_wrappers.append(name)
        self.safe(write_json, self.directory / 'observer_status.json', {
            'diagnostics_version': DIAGNOSTICS_VERSION,
            'installed_hooks': [name for _, name, _, _ in self.hooks], 'warnings': self.warnings,
            'retained_runtime_wrappers': self.retained_runtime_wrappers,
            'requests': len(self.requests), 'events': len(self.events)})


def usage_summary(rows):
    rows = list(rows)
    report = {'model_requests': len(rows)}
    for metric in ['input_tokens', 'output_tokens', 'total_tokens', 'cached_tokens', 'reasoning_tokens',
                   'model_request_wall_seconds', 'model_generator_active_seconds']:
        observed = [row[metric] for row in rows if number(row.get(metric)) is not None]
        report[metric + '_observed_sum'] = sum(observed) if observed else None
        report[metric + '_coverage'] = len(observed)
        report[metric] = sum(observed) if rows and len(observed) == len(rows) else None
    return report


def grouped_usage(rows, keys):
    groups = defaultdict(list)
    for row in rows:
        groups[tuple(row.get(key) or 'unknown' for key in keys)].append(row)
    return [{**dict(zip(keys, group)), **usage_summary(subset)} for group, subset in sorted(groups.items())]


def interaction_rows(events, helpers=()):
    """Flatten observed exchanges without inventing parent links or tool usage."""
    rows = []
    for event in events:
        common = {k: event.get(k) for k in ('observer_elapsed_seconds', 'agent', 'agent_run_id',
                                           'parent_agent_run_id', 'parent_tool_call_id', 'event_id', 'invocation_id')}
        if event['kind'] != 'agent_event':
            rows.append({**common, **event})
            continue
        for call in event.get('calls', []):
            rows.append({**common, 'kind': 'delegation_request' if call['function_name'] in helpers else 'tool_call',
                         'from': event['agent'], 'to': call['function_name'], **call})
        for response in event.get('responses', []):
            rows.append({**common, 'kind': 'delegation_response' if response['function_name'] in helpers else 'tool_result',
                         'from': response['function_name'], 'to': event['agent'], **response})
        if event.get('visible_text'):
            rows.append({**common, 'kind': 'agent_answer', 'from': event['agent'], 'text': event['visible_text']})
        if event.get('error_code') or event.get('error_message'):
            rows.append({**common, 'kind': 'agent_error', 'error_code': event.get('error_code'),
                         'error_message': event.get('error_message')})
    return rows


def observed_tools(interactions):
    """Match native IDs within a caller run; leave unmatched outcomes unknown."""
    tools, pending = [], {}
    for row in interactions:
        key = (row.get('agent_run_id'), row.get('tool_call_id'))
        if row['kind'] in {'tool_call', 'delegation_request'}:
            tool = {**row, 'tool_name': row.get('function_name'), 'result': None, 'status': None,
                    'start_observer_seconds': row['observer_elapsed_seconds'], 'tool_wall_seconds': None,
                    'call_outcome': 'no_response_observed'}
            tools.append(tool)
            if key[1] is not None:
                pending[key] = tool
        elif row['kind'] in {'tool_result', 'delegation_response'} and key[1] is not None and key in pending:
            tool = pending.pop(key)
            result = decode_result(row.get('response'))
            status = value(result, 'status')
            if status is None and value(result, 'error'):
                status = 'error'
            tool.update(result=result, status=status,
                        call_outcome='response_observed',
                        end_observer_seconds=row['observer_elapsed_seconds'],
                        tool_wall_seconds=row['observer_elapsed_seconds'] - tool['start_observer_seconds'])
    return tools


def observed_agent_runs(events):
    runs = {}
    for event in events:
        if event['kind'] == 'agent_start':
            runs[event['agent_run_id']] = {**event, 'start_observer_seconds': event['observer_elapsed_seconds'],
                                          'inclusive_wall_seconds': None, 'end_observer_seconds': None}
        elif event['kind'] == 'agent_end' and event['agent_run_id'] in runs:
            runs[event['agent_run_id']].update(end_observer_seconds=event['observer_elapsed_seconds'],
                inclusive_wall_seconds=event.get('inclusive_wall_seconds'), status=event.get('status'))
    return list(runs.values())


def request_health(requests, trace=None):
    """Reconcile observed usage without treating missing usage as zero."""
    summary = usage_summary(requests)
    reconciliation = {}
    for label, metric in [('input', 'input_tokens'), ('output', 'output_tokens'), ('total', 'total_tokens')]:
        reported = value(value(trace or {}, 'final_metrics', {}), 'total_prompt_tokens' if label == 'input'
                         else 'total_completion_tokens' if label == 'output' else 'total_tokens')
        observed = summary[metric + '_observed_sum']
        reconciliation[label] = {'atif_reported': reported, 'observed_sum': observed,
                                'difference': observed - reported if number(observed) is not None and number(reported) is not None else None}
    return {'model_requests_observed': len(requests),
            'requests_with_token_usage': sum(number(r.get('input_tokens')) is not None and number(r.get('output_tokens')) is not None for r in requests),
            'requests_without_usage': sum(number(r.get('input_tokens')) is None or number(r.get('output_tokens')) is None for r in requests),
            'request_status_counts': dict(Counter(r.get('status', 'not_exported') for r in requests)),
            'atif_reconciliation': reconciliation}


def write_interaction_text(path, rows):
    lines = ['Visible agent/tool interaction timeline. Full arguments/results remain in JSON.', '']
    for row in rows:
        stamp = row.get('observer_elapsed_seconds', row.get('elapsed_seconds'))
        lines.append(f"{stamp if stamp is not None else '?'}s {row.get('kind')} {row.get('from', row.get('agent', '?'))} -> {row.get('to', '')}")
        for key in ('arguments', 'response', 'result', 'text'):
            if row.get(key) is not None:
                text = json.dumps(row[key], ensure_ascii=False)
                lines.append(f"  {key}: " + text[:1200] + (' [preview truncated; see JSON]' if len(text) > 1200 else ''))
    Path(path).write_text('\n'.join(lines) + '\n', encoding='utf-8')


def submission_metadata(agent_dir):
    """Read includes without importing submission code; hash config/prompt files."""
    import yaml
    root = Path(agent_dir).resolve()
    agents, helpers, models, hashes = {}, set(), set(), {}
    def load(path, stack=()):
        path = path.resolve()
        if not path.is_relative_to(root) or path in stack:
            raise ValueError('Invalid or cyclic configuration include')
        text = path.read_text(encoding='utf-8')
        hashes[path.relative_to(root).as_posix()] = hashlib.sha256(path.read_bytes()).hexdigest()
        if path.suffix.lower() not in {'.yaml', '.yml'}:
            return text
        class Loader(yaml.SafeLoader):
            pass
        Loader.add_constructor('!include', lambda loader, node: load(path.parent / loader.construct_scalar(node), (*stack, path)))
        return yaml.load(text, Loader=Loader)
    def visit(config):
        if not isinstance(config, dict):
            return
        if 'name' in config:
            name = config['name']
            models.add(config.get('model'))
            agents[name] = {k: config.get(k) for k in ('model', 'adapter', 'agent_class', 'generate_content_config', 'include_contents')}
        for child in config.get('sub_agents', []) or []:
            visit(load(root / child['config_path']) if isinstance(child, dict) and 'config_path' in child else child)
        for tool in config.get('tools', []) or []:
            if isinstance(tool, dict) and 'agent_tool' in tool:
                ref = tool['agent_tool']
                child = load(root / ref['config_path']) if 'config_path' in ref else ref
                if isinstance(child, dict) and child.get('name'):
                    helpers.add(child['name'])
                visit(child)
    config = load(root / 'agent.yaml')
    visit(config)
    for name in ('eval_config.yaml', 'eval_config.yml'):
        if (root / name).is_file():
            load(root / name)
    return {'root_agent': value(config, 'name'), 'agents': agents, 'helper_names': sorted(helpers),
            'declared_models': sorted(m for m in models if m), 'file_sha256': hashes,
            'config_sha256': hashlib.sha256(json.dumps(hashes, sort_keys=True).encode()).hexdigest()}


def snapshot_submission(agent_dir, destination, metadata):
    """Retain the exact active configuration and prompt bytes already hashed."""
    root, destination = Path(agent_dir).resolve(), Path(destination).resolve()
    destination.mkdir(parents=True, exist_ok=True)
    for name, expected in metadata['file_sha256'].items():
        source, target = (root / name).resolve(), (destination / name).resolve()
        if not source.is_relative_to(root) or not target.is_relative_to(destination):
            raise ValueError('Configuration snapshot path escapes its root')
        content = source.read_bytes()
        if hashlib.sha256(content).hexdigest() != expected:
            raise ValueError(f'Configuration changed during snapshot: {name}')
        if target.exists() and target.read_bytes() != content:
            raise ValueError('Use a fresh results directory for a different configuration')
        target.parent.mkdir(parents=True, exist_ok=True)
        target.write_bytes(content)
    write_json(destination / 'snapshot_manifest.json', metadata)


def runtime_versions():
    versions = {}
    for name in ('swegemma', 'adk-submission', 'adk-eval-core', 'google-adk', 'vllm', 'litellm', 'torch'):
        try:
            versions[name] = importlib.metadata.version(name)
        except importlib.metadata.PackageNotFoundError:
            versions[name] = None
    return versions


def inference_metadata(namespace):
    """Record available serving settings without dumping model registries/keys."""
    config = namespace.get('vllm_cfg') or namespace.get('vllm_config')
    return {key: plain(value(config, key)) for key in
            ('model', 'max_model_len', 'gpu_memory_utilization', 'tensor_parallel_size', 'dtype', 'quantization')}


def run_sync(coro_or_fn, *args, **kwargs):
    fn = (lambda: coro_or_fn(*args, **kwargs)) if callable(coro_or_fn) else (lambda: coro_or_fn)
    try:
        loop = asyncio.get_running_loop()
    except RuntimeError:
        loop = None
    if loop is not None and loop.is_running():
        with concurrent.futures.ThreadPoolExecutor(max_workers=1) as pool:
            context = contextvars.copy_context()
            return pool.submit(lambda: context.run(lambda: asyncio.run(fn()))).result()
    return asyncio.run(fn())


class Tee(io.TextIOBase):
    def __init__(self, original, saved):
        self.original, self.saved = original, saved
    def write(self, text):
        self.saved.write(text)
        return self.original.write(text)
    def flush(self):
        self.saved.flush()
        self.original.flush()
    def isatty(self):
        return False


def evaluate_with_diagnostics(evaluator, tasks, results_dir, agent_dir, task_types=None, variant='candidate', live_observer=True):
    """Preserve each outcome before starting the next task; continue ordinary errors."""
    import sys
    tasks = list(tasks)
    identifiers = [str(value(task, 'instance_id')) for task in tasks]
    if len(set(identifiers)) != len(identifiers) or any(not re.fullmatch(r'[\w.-]+', i) or i in {'.', '..'} for i in identifiers):
        raise ValueError('Task IDs must be unique, safe filenames')
    root = Path(results_dir)
    root.mkdir(parents=True, exist_ok=True)
    for directory in ('raw_results', 'patches', 'logs', 'console', 'observer', 'report', 'test_outputs'):
        (root / directory).mkdir(exist_ok=True)
    metadata = submission_metadata(agent_dir)
    snapshot_submission(agent_dir, root / 'configuration_snapshot', metadata)
    write_json(root / 'submission_metadata.json', metadata)
    public_tasks = [{key: plain(value(task, key)) for key in ('instance_id', 'repo', 'base_commit', 'problem_statement')} for task in tasks]
    write_json(root / 'selected_tasks.json', public_tasks)
    helpers = metadata['helper_names']
    records = []
    for idx, task in enumerate(tasks, 1):
        identifier = str(value(task, 'instance_id'))
        print(f'[{variant} {idx}/{len(tasks)}] Evaluating {identifier} ({value(task, "repo")})')
        record = {'instance_id': identifier, 'repo': value(task, 'repo'), 'variant': variant,
                  'task_type': (task_types or {}).get(identifier, 'unclassified'), 'resolved': None,
                  'test_exit_code': None, 'agent_patch': None, 'patch_chars': None,
                  'tool_calls': None, 'duration_seconds': None, 'stop_reason': None, 'error': None,
                  'status': 'running', 'started_at_utc': datetime.now(timezone.utc).isoformat(),
                  'diagnostics_version': DIAGNOSTICS_VERSION}
        statement = value(task, 'problem_statement', '') or ''
        record['task_title'] = next((line.strip()[:200] for line in statement.splitlines() if line.strip()), None)
        append_json(root / 'task_events.jsonl', record)
        observer = DiagnosticObserver(root / 'observer' / identifier, helpers, enabled=live_observer, instance_id=identifier)
        started = time.perf_counter()
        result_returned = False
        root_logger = logging.getLogger()
        handler = logging.FileHandler(root / 'logs' / f'observer_{identifier}.log', encoding='utf-8')
        handler.setLevel(logging.INFO)
        handler.setFormatter(logging.Formatter('%(asctime)s %(levelname)s %(name)s %(message)s'))
        handler.addFilter(lambda entry: entry.name.startswith(('swegemma', 'adk_eval_core', 'adk_submission', 'google.adk')))
        old_level = root_logger.level
        root_logger.setLevel(min(old_level, logging.INFO))
        root_logger.addHandler(handler)
        try:
            with (root / 'console' / f'{identifier}.txt').open('w', encoding='utf-8') as transcript:
                with contextlib.redirect_stdout(Tee(sys.stdout, transcript)), contextlib.redirect_stderr(Tee(sys.stderr, transcript)), observer:
                    result = run_sync(evaluator.evaluate_task, task=task, task_index=idx, total_tasks=len(tasks))
            result_returned = True
            for key in ('resolved', 'test_exit_code', 'agent_patch', 'tool_calls', 'duration_seconds',
                        'agent_elapsed_seconds', 'error', 'error_type', 'stop_reason', 'exit_status', 'stage'):
                field = value(result, key)
                if field is not None:
                    record[key] = plain(field)
            if record['error'] is None:
                record['error'] = value(result, 'error_message')
            if record['stop_reason'] is not None:
                record['stop_reason_source'] = 'result.stop_reason'
            elif isinstance(record['error'], str) and re.match(r'^Agent exceeded session timeout\b', record['error']):
                record['stop_reason'] = 'session_timeout'
                record['stop_reason_source'] = 'explicit_harness_error'
            record['status'] = 'finished'
            # Persist review-critical artifacts before serializing the full runtime object.
            if isinstance(record['agent_patch'], str):
                record['patch_chars'] = len(record['agent_patch'])
                (root / 'patches' / f'{identifier}_returned.patch').write_text(record['agent_patch'], encoding='utf-8')
            test_output = value(result, 'test_output')
            record['verification_output_path'] = None
            if isinstance(test_output, str) and test_output:
                test_path = root / 'test_outputs' / f'{identifier}_returned.log'
                test_path.write_text(test_output, encoding='utf-8')
                record['verification_output_path'] = str(test_path.relative_to(root))
            raw = plain(result)
            record['raw_result'] = raw
            write_json(root / 'raw_results' / f'{identifier}.json', raw)
        except BaseException as error:
            if result_returned and isinstance(error, Exception):
                record['status'] = 'finished'
                record.setdefault('diagnostic_errors', []).append({'stage': 'save_result',
                    'error': str(error), 'error_type': type(error).__name__})
            else:
                record.update(status='error' if isinstance(error, Exception) else 'interrupted',
                              error=str(error), error_type=type(error).__name__)
            (root / 'console' / f'{identifier}_exception.txt').write_text(traceback.format_exc(), encoding='utf-8')
            if not isinstance(error, Exception):
                raise
        finally:
            root_logger.removeHandler(handler)
            root_logger.setLevel(old_level)
            handler.close()
            record['observer_wall_seconds'] = time.perf_counter() - started
            record['finished_at_utc'] = datetime.now(timezone.utc).isoformat()
            trace_path = root / 'traces' / f'trace_{identifier}.json'
            trace = {}
            trace_warnings = []
            if trace_path.is_file():
                try:
                    trace = describe_trace(json.loads(trace_path.read_text(encoding='utf-8')), helpers)
                except (OSError, ValueError, TypeError) as error:
                    trace_warnings.append(f'Trace read failed: {error}')
            requests = observer.requests or trace.get('requests', [])
            for row in requests:
                row.update(instance_id=identifier, variant=variant, task_type=record['task_type'])
            record.update(usage_source='live_adk' if observer.requests else 'atif' if trace else 'unavailable',
                          **usage_summary(requests), trace_path=str(trace_path) if trace_path.is_file() else None,
                          warnings=observer.warnings + trace.get('warnings', []) + trace_warnings)
            record['warnings'].extend(f"Diagnostic {issue['stage']} failed: {issue['error_type']}: {issue['error']}"
                                      for issue in record.get('diagnostic_errors', []))
            record['observed_trace_window_seconds'] = trace.get('observed_trace_window_seconds')
            health = request_health(requests, trace)
            record['requests_without_usage'] = health['requests_without_usage']
            record['observed_usage_complete'] = bool(requests) and health['requests_without_usage'] == 0
            for helper in helpers:
                delegated = any(c['function_name'] == helper for event in observer.events for c in event.get('calls', []))
                delegated |= any(t['tool_name'] == helper for t in trace.get('tools', []))
                if delegated and not any(r['agent'] == helper for r in requests):
                    record['warnings'].append(f'{helper}: internal model requests not observed; per-helper usage incomplete')
            # ATIF and observer clocks are separate; do not splice their timestamps.
            interactions = interaction_rows(observer.events, helpers) if observer.events else trace.get('timeline', [])
            write_json(root / 'report' / f'{identifier}_interactions.json', interactions)
            write_csv(root / 'report' / f'{identifier}_interactions.csv', interactions)
            write_interaction_text(root / 'report' / f'{identifier}_interaction_log.txt', interactions)
            write_csv(root / 'report' / f'{identifier}_requests.csv', requests)
            write_json(root / 'report' / f'{identifier}_requests.json', requests)
            tool_rows = observed_tools(interactions) if observer.events else trace.get('tools', [])
            record['pending_tool_calls'] = sum(t.get('call_outcome', 'response_observed' if t.get('result') is not None else 'no_response_observed') == 'no_response_observed' for t in tool_rows)
            health['pending_tool_calls'] = record['pending_tool_calls']
            health['console_stdout_bytes'] = (root / 'console' / f'{identifier}.txt').stat().st_size
            write_json(root / 'report' / f'{identifier}_measurement_health.json', health)
            runs = observed_agent_runs(observer.events)
            write_csv(root / 'report' / f'{identifier}_agent_runs.csv', runs)
            write_json(root / 'report' / f'{identifier}_agent_runs.json', runs)
            write_json(root / 'report' / f'{identifier}_tools.json', tool_rows)
            write_csv(root / 'report' / f'{identifier}_tools.csv', tool_rows)
            append_json(root / 'notebook_task_results.jsonl', record)
            records.append(record)
            save_aggregate_reports(root, records, len(tasks))
        print(f"  -> resolved={record['resolved']}, patch_chars={record['patch_chars']}, "
              f"tools={record['tool_calls']}, duration={record['duration_seconds']}, "
              f"input={token_display(record, 'input_tokens')}, output={token_display(record, 'output_tokens')}, "
              f"stop={record['stop_reason'] or 'not recorded'}")
    return records


def token_display(record, metric):
    if number(record.get(metric)) is not None:
        return str(record[metric])
    partial = record.get(metric + '_observed_sum')
    return f">={partial} ({record.get(metric + '_coverage', 0)}/{record.get('model_requests', 0)} requests)" if partial is not None else 'unknown'


def save_aggregate_reports(root, records, selected_count):
    root = Path(root)
    requests = []
    for record in records:
        path = root / 'report' / f"{record['instance_id']}_requests.json"
        if path.is_file():
            requests.extend(json.loads(path.read_text(encoding='utf-8')))
    write_csv(root / 'report' / 'requests.csv', requests)
    for name, keys in [('agent_usage', ['instance_id', 'agent']), ('stage_usage', ['instance_id', 'stage']),
                       ('agent_stage_usage', ['instance_id', 'agent', 'stage'])]:
        rows = grouped_usage(requests, keys)
        write_json(root / 'report' / f'{name}.json', rows)
        write_csv(root / 'report' / f'{name}.csv', rows)
    fields = ['instance_id', 'repo', 'task_title', 'task_type', 'status', 'resolved', 'test_exit_code', 'patch_chars',
              'tool_calls', 'duration_seconds', 'observer_wall_seconds', 'agent_elapsed_seconds', 'stop_reason',
              'stop_reason_source',
              'usage_source', 'model_requests', 'input_tokens', 'output_tokens', 'total_tokens',
              'input_tokens_observed_sum', 'output_tokens_observed_sum', 'reasoning_tokens', 'cached_tokens',
              'total_tokens_observed_sum', 'input_tokens_coverage', 'output_tokens_coverage',
              'requests_without_usage', 'observed_usage_complete', 'pending_tool_calls', 'verification_output_path',
              'model_request_wall_seconds', 'model_generator_active_seconds', 'error', 'diagnostic_errors', 'warnings']
    write_csv(root / 'report' / 'task_summary.csv', records, fields)
    groups = defaultdict(list)
    for record in records:
        groups[record['task_type']].append(record)
    grouped = []
    for task_type, rows in sorted(groups.items()):
        known = [r for r in rows if isinstance(r['resolved'], bool)]
        durations = [r['duration_seconds'] for r in rows if number(r['duration_seconds']) is not None]
        missing_usage = sum(row['usage_source'] == 'unavailable' for row in rows)
        group_usage = usage_summary([r for r in requests if r['instance_id'] in {t['instance_id'] for t in rows}])
        if missing_usage:
            for metric in ('input_tokens', 'output_tokens', 'total_tokens', 'cached_tokens', 'reasoning_tokens',
                           'model_request_wall_seconds', 'model_generator_active_seconds'):
                group_usage[metric] = None
        grouped.append({'task_type': task_type, 'tasks': len(rows), 'known_outcomes': len(known),
                        'repositories': sorted({row['repo'] or 'unknown' for row in rows}), 'tasks_without_usage': missing_usage,
                        'resolved': sum(r['resolved'] for r in known),
                        'resolution_rate': sum(r['resolved'] for r in known) / len(rows) if len(known) == len(rows) else None,
                        'median_total_task_seconds': statistics.median(durations) if durations else None,
                        **group_usage})
    write_csv(root / 'report' / 'task_type_summary.csv', grouped)
    summary = {'selected_tasks': selected_count, 'attempted_tasks': len(records),
               'unstarted_tasks': selected_count - len(records),
               'known_outcomes': sum(isinstance(r['resolved'], bool) for r in records),
               'resolved': sum(r['resolved'] is True for r in records),
               'unresolved': sum(r['resolved'] is False for r in records),
               'outcomes_unknown': sum(r['resolved'] is None for r in records),
               'unclassified_tasks': sum(r['task_type'] == 'unclassified' for r in records),
               'tasks_with_missing_request_usage': sum(r.get('requests_without_usage', 0) > 0 for r in records),
               'tasks_with_diagnostic_errors': sum(bool(r.get('diagnostic_errors')) for r in records),
               'pending_tool_calls_observed': sum(r.get('pending_tool_calls', 0) for r in records),
               'resolution_rate': sum(r['resolved'] is True for r in records) / selected_count
               if selected_count and len(records) == selected_count and all(isinstance(r['resolved'], bool) for r in records) else None,
               'measurement_notes': [
                   'Missing usage stays null; observed sums and coverage expose partial measurements.',
                   'Live model observations are preferred; ATIF fallback is never added to live counts.',
                   'Stages are inferred from agent roles and actions; unknown/mixed stages remain explicit.',
                   'Cached/reasoning usage follows provider semantics; these fields are never added to reported total tokens.',
                   'Model generator active time measures awaits in the SDK generator, including serving/queueing/retries.',
                   'Model request wall time ends at the last response chunk; streaming wall time can include consumer pauses.',
                   'Parallel/nested durations overlap; agent run times are inclusive and cannot be summed into task time.',
                   'ATIF elapsed times and observer elapsed times use separate clocks.',
                   'A missing stop reason is not inferred from duration or empty patch.',
                   'Task types are user-supplied labels; unclassified is the default.',
               ]}
    write_json(root / 'report' / 'summary.json', summary)
    write_json(root / 'report' / 'task_summary.json', records)
    return summary


def evaluation_kwargs(config_class, settings):
    """Support older wheelhouses while recording unsupported verification timeout."""
    signature = inspect.signature(config_class)
    parameters = signature.parameters
    supports = 'verification_timeout_seconds' in parameters or any(p.kind == p.VAR_KEYWORD for p in parameters.values())
    kwargs = dict(settings)
    warnings = []
    if not supports:
        kwargs.pop('verification_timeout_seconds', None)
        warnings.append('Installed EvalConfig lacks verification_timeout_seconds; verification timeout is runtime-dependent')
    return kwargs, warnings


def archive_diagnostics(results_dir):
    root = Path(results_dir)
    return Path(shutil.make_archive(str(root), 'zip', root_dir=root.parent, base_dir=root.name))


def audit_diagnostics_archive(archive_path):
    """Read-only checks against archived evidence; no model/runtime required."""
    errors, warnings, rows = [], [], []
    with zipfile.ZipFile(archive_path) as archive:
        manifests = [n for n in archive.namelist() if n == 'run_manifest.json' or n.endswith('/run_manifest.json')]
        if len(manifests) != 1:
            raise ValueError('Expected exactly one run manifest in the archive')
        prefix = manifests[0][:-len('run_manifest.json')]
        names = set(archive.namelist())
        def read(name):
            return archive.read(prefix + name).decode('utf-8')
        def exists(name):
            return prefix + name in names
        manifest = json.loads(read('run_manifest.json'))
        summary = json.loads(read('report/summary.json'))
        tasks = json.loads(read('report/task_summary.json'))
        if archive.testzip() is not None:
            errors.append('ZIP integrity check failed')
        if len({t['instance_id'] for t in tasks}) != len(tasks):
            errors.append('Duplicate task summary IDs')
        if summary['resolved'] != sum(t.get('resolved') is True for t in tasks):
            errors.append('Resolved count disagrees with task outcomes')
        for task in tasks:
            identifier = task['instance_id']
            patch = task.get('agent_patch')
            patch_chars = len(patch) if isinstance(patch, str) else task.get('patch_chars')
            if isinstance(patch, str) and task.get('patch_chars') != patch_chars:
                errors.append(f'{identifier}: patch_chars disagrees with the returned patch ({patch_chars} chars)')
            if task.get('status') in {'error', 'interrupted'}:
                warnings.append(f"{identifier}: evaluation/diagnostics status={task['status']}: {task.get('error') or 'reason not recorded'}")
            if task.get('diagnostic_errors'):
                warnings.append(f'{identifier}: result returned but diagnostic artifact capture was incomplete')
            request_path = f'observer/{identifier}/requests.jsonl'
            raw = [json.loads(line) for line in read(request_path).splitlines() if line.strip()] if exists(request_path) else []
            report = json.loads(read(f'report/{identifier}_requests.json'))
            status_path = f'observer/{identifier}/observer_status.json'
            status = json.loads(read(status_path)) if exists(status_path) else {}
            if task.get('usage_source') == 'live_adk':
                if len(raw) != len(report) or status.get('requests') != len(raw):
                    errors.append(f'{identifier}: raw/status/report request counts disagree (possible cross-task leak)')
                keys = ('request_id', 'agent', 'input_tokens', 'output_tokens', 'total_tokens', 'status')
                if [tuple(r.get(k) for k in keys) for r in raw] != [tuple(r.get(k) for k in keys) for r in report]:
                    errors.append(f'{identifier}: raw and report requests differ')
                if any(r.get('instance_id') not in (None, identifier) for r in raw):
                    errors.append(f'{identifier}: request attributed to a different task')
            metrics = usage_summary(report)
            for metric in ('input_tokens', 'output_tokens', 'total_tokens'):
                if task.get(metric + '_observed_sum') != metrics[metric + '_observed_sum']:
                    errors.append(f'{identifier}: {metric} observed sum disagrees with requests')
            trace = json.loads(read(f'traces/trace_{identifier}.json')) if exists(f'traces/trace_{identifier}.json') else {}
            trace_detail = describe_trace(trace, manifest.get('submission', {}).get('helper_names', []))
            health = request_health(report, trace_detail)
            if any(c['difference'] not in (None, 0) for c in health['atif_reconciliation'].values()):
                warnings.append(f'{identifier}: live/ATIF observed token totals differ; check additional helper or compaction requests')
            if health['requests_without_usage']:
                warnings.append(f'{identifier}: {health["requests_without_usage"]} observed request(s) lack usage; totals are lower bounds')
            if any(r.get('error_type') == 'GeneratorExit' and r.get('response_chunks', 0) for r in raw):
                warnings.append(f'{identifier}: legacy request status labels consumer closure after a response as interrupted')
            tools = json.loads(read(f'report/{identifier}_tools.json'))
            pending = sum(t.get('call_outcome', 'response_observed' if t.get('result') is not None else 'no_response_observed') == 'no_response_observed' for t in tools)
            helpers = manifest.get('submission', {}).get('helper_names', [])
            delegation = sum(t.get('tool_name') in helpers for t in tools)
            if pending:
                warnings.append(f'{identifier}: {pending} tool call(s) have no recorded result')
            warnings.extend(f'{identifier}: observer warning: {warning}' for warning in status.get('warnings', []))
            raw_result_path = f'raw_results/{identifier}.json'
            if exists(raw_result_path) and json.loads(read(raw_result_path)).get('resolved') != task.get('resolved'):
                errors.append(f'{identifier}: raw and normalized outcomes differ')
            if isinstance(task.get('resolved'), bool):
                if not exists(raw_result_path):
                    warnings.append(f'{identifier}: raw returned result artifact is missing')
                if isinstance(patch, str) and not exists(f'patches/{identifier}_returned.patch'):
                    warnings.append(f'{identifier}: returned patch artifact is missing')
                if number(task.get('test_exit_code')) is not None and task['test_exit_code'] >= 0 and not task.get('verification_output_path'):
                    warnings.append(f'{identifier}: verification exit recorded but returned verification output is unavailable')
            rows.append({'instance_id': identifier, 'resolved': task.get('resolved'), 'patch_chars': patch_chars,
                         'reported_patch_chars': task.get('patch_chars'),
                         'duration_seconds': task.get('duration_seconds'), 'stop_reason': task.get('stop_reason'),
                         'model_requests': len(report), 'usage_requests': health['requests_with_token_usage'],
                         'observed_tokens': metrics['total_tokens_observed_sum'], 'pending_tools': pending,
                         'helper_calls': delegation, 'request_agents': sorted({r.get('agent', 'unknown') for r in report}),
                         'atif_reconciliation': health['atif_reconciliation']})
        snapshots = [n for n in names if n.startswith(prefix + 'configuration_snapshot/')]
        if not snapshots:
            warnings.append('Configuration/prompt snapshot is absent; only hashes and declared settings are retained')
        else:
            for name, expected in manifest.get('submission', {}).get('file_sha256', {}).items():
                relative = 'configuration_snapshot/' + name
                if not exists(relative) or hashlib.sha256(archive.read(prefix + relative)).hexdigest() != expected:
                    errors.append(f'Configuration snapshot does not match manifest: {name}')
        if any(t.get('task_type') == 'unclassified' for t in tasks):
            warnings.append('Task-type comparisons need labels assigned before testing')
        coverage = {
            'helper_invocation_observed': any(r['helper_calls'] for r in rows),
            'nonempty_patch_observed': any((r['patch_chars'] or 0) > 0 for r in rows),
            'verification_exit_observed': any(number(t.get('test_exit_code')) is not None and t['test_exit_code'] >= 0 for t in tasks),
            'solved_task_observed': any(t.get('resolved') is True for t in tasks),
            'configuration_snapshot_present': bool(snapshots),
        }
    return {'archive': str(archive_path), 'diagnostics_version': manifest.get('diagnostics_version'),
            'integrity_errors': errors, 'warnings': warnings, 'tasks': rows, 'validation_coverage': coverage,
            'effective_settings': manifest.get('effective_settings_passed_to_evaluator'),
            'declared_models': manifest.get('submission', {}).get('declared_models')}

In [8]:

# Replacement evaluation cell begins here.

import tempfile
import pandas as pd
import yaml
from google.adk.agents.context_cache_config import ContextCacheConfig
from google.adk.apps._configs import EventsCompactionConfig
from swegemma.budget import EvaluationBudget, HarnessLimits
from swegemma.config import EvalConfig, build_submission_limits
from swegemma.evaluate import Evaluator

eval_config_file = next((AGENT_DIR / name for name in ('eval_config.yaml', 'eval_config.yml')
                         if (AGENT_DIR / name).is_file()), None)
raw_eval_cfg = yaml.safe_load(eval_config_file.read_text(encoding='utf-8')) or {} if eval_config_file else {}
if not isinstance(raw_eval_cfg, dict):
    raise ValueError('Evaluation config must be a mapping')
eval_section = raw_eval_cfg.get('evaluation', raw_eval_cfg)
if not isinstance(eval_section, dict):
    raise ValueError('Evaluation settings must be a mapping')
default_budget, default_harness = EvaluationBudget(), HarnessLimits()
defaults = {
    'max_time_minutes': default_budget.time_minutes,
    'max_tool_calls': 100,
    'max_turns': default_budget.turns,
    'max_total_tokens': default_budget.total_tokens,
    'max_cost_usd': default_budget.cost_usd,
    'timeout_seconds': default_harness.command_timeout_seconds,
    'verification_timeout_seconds': getattr(default_harness, 'verification_timeout_seconds', 300),
    'max_stdout_chars': default_harness.max_stdout_chars,
    'max_file_lines': default_harness.max_file_lines,
    'max_file_chars': default_harness.max_file_chars,
}
unknown = set(EVAL_OVERRIDES) - defaults.keys()
if unknown:
    raise ValueError(f'Unsupported evaluation overrides: {sorted(unknown)}')
eval_section = {**eval_section, **{key: val for key, val in EVAL_OVERRIDES.items() if val != INHERIT}}
if 'max_turns' not in eval_section and 'max_llm_calls' in eval_section:
    eval_section['max_turns'] = eval_section['max_llm_calls']
effective_limits = {key: eval_section.get(key, default) for key, default in defaults.items()}
for key, val in effective_limits.items():
    if val is not None and (number(val) is None or val <= 0):
        raise ValueError(f'{key} must be a positive finite number or None')
    if val is not None and key not in {'max_time_minutes', 'max_cost_usd'} and not isinstance(val, int):
        raise ValueError(f'{key} must be a positive integer or None')
budget = EvaluationBudget(
    time_minutes=effective_limits['max_time_minutes'], tool_calls=effective_limits['max_tool_calls'],
    turns=effective_limits['max_turns'], total_tokens=effective_limits['max_total_tokens'],
    cost_usd=effective_limits['max_cost_usd'],
)
# Passing max_time_minutes=None alone would inherit EvaluationBudget's 60 minutes.
# Set the structured budget as well, so None actually disables the session timer.
harness_values = {
    'command_timeout_seconds': effective_limits['timeout_seconds'],
    'verification_timeout_seconds': effective_limits['verification_timeout_seconds'],
    'max_stdout_chars': effective_limits['max_stdout_chars'],
    'max_file_lines': effective_limits['max_file_lines'],
    'max_file_chars': effective_limits['max_file_chars'],
}
harness_parameters = inspect.signature(HarnessLimits).parameters
config_warnings = []
if 'verification_timeout_seconds' not in harness_parameters and not any(
    p.kind == p.VAR_KEYWORD for p in harness_parameters.values()
):
    harness_values.pop('verification_timeout_seconds')
    config_warnings.append('Installed HarnessLimits lacks a separate verification timeout')
harness = HarnessLimits(**harness_values)
settings, compatibility_warnings = evaluation_kwargs(EvalConfig, {
    key: effective_limits[key] for key in ('timeout_seconds', 'verification_timeout_seconds',
                                         'max_tool_calls', 'max_time_minutes', 'max_turns')
})
config_warnings.extend(compatibility_warnings)
if TASK_IDS:
    indexed = {task.instance_id: task for task in tasks}
    if len(set(TASK_IDS)) != len(TASK_IDS) or set(TASK_IDS) - indexed.keys():
        raise ValueError('TASK_IDS contains duplicates or unknown task IDs')
    SAMPLE_TASKS = [indexed[identifier] for identifier in TASK_IDS]
else:
    if TASK_LIMIT is not None and (not isinstance(TASK_LIMIT, int) or isinstance(TASK_LIMIT, bool) or TASK_LIMIT <= 0):
        raise ValueError('TASK_LIMIT must be a positive integer or None')
    SAMPLE_TASKS = tasks if TASK_LIMIT is None else tasks[:TASK_LIMIT]
if not SAMPLE_TASKS:
    raise ValueError('No tasks selected')


### landing the records
run_parent = WORKING_DIR / 'diagnostic_runs'
run_parent.mkdir(parents=True, exist_ok=True)
#prefix = datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%S.%fZ_')
RESULTS_DIR = Path(tempfile.mkdtemp(prefix=submission_x, dir=run_parent))
limits, gen_constraints = build_submission_limits()
eval_config = EvalConfig(
    tasks_path=TASKS_PATH, snapshots_dir=DATA_DIR / 'snapshots',
    results_dir=RESULTS_DIR, submission_dir=AGENT_DIR, models=models,
    sandbox='subprocess', budget=budget, harness=harness, **settings,
    limits=limits, generation_constraints=gen_constraints, adapter_manifest=adapters,
    context_cache_config=ContextCacheConfig(**CACHE_SETTINGS),
    events_compaction_config=EventsCompactionConfig(**COMPACTION_SETTINGS),
    graph_dir=GRAPH_DIR, embeddings_dir=EMBEDDINGS_DIR, wheels_dir=DATA_DIR / 'wheels',
    verbose=False,
)
if value(value(eval_config, 'budget'), 'time_minutes') != effective_limits['max_time_minutes']:
    raise RuntimeError('EvalConfig did not retain the requested per-task time budget')
metadata = submission_metadata(AGENT_DIR)
write_json(RESULTS_DIR / 'run_manifest.json', {
    'started_at_utc': datetime.now(timezone.utc).isoformat(),
    'task_ids': [task.instance_id for task in SAMPLE_TASKS], 'task_types': TASK_TYPES,
    'effective_settings_passed_to_evaluator': settings, 'eval_overrides': EVAL_OVERRIDES,
    'effective_evaluation_limits': effective_limits,
    'effective_budget': plain(value(eval_config, 'budget')),
    'effective_harness_limits': plain(value(eval_config, 'harness')),
    'session_time_limit_disabled': effective_limits['max_time_minutes'] is None,
    'configuration_warnings': config_warnings, 'submission': metadata,
    'runtime_versions': runtime_versions(), 'sandbox': 'subprocess',
    'observer_enabled': LIVE_AGENT_OBSERVER,
    'diagnostics_version': DIAGNOSTICS_VERSION,
    'inference_server_settings': inference_metadata(globals()),
    'context_cache': CACHE_SETTINGS,
    'compaction': COMPACTION_SETTINGS,
})
print('Effective evaluation limits:', effective_limits)
print('Declared models:', metadata['declared_models'], 'helpers:', metadata['helper_names'])
print('Artifacts:', RESULTS_DIR)
for warning in config_warnings:
    print('Configuration warning:', warning)
if effective_limits['max_time_minutes'] is None:
    print('Per-task session time limit: DISABLED. Other limits above still apply.')
elif effective_limits['max_time_minutes'] <= 1:
    print('This run has at most ONE MINUTE of agent time per task, as configured.')

evaluator = Evaluator(eval_config)
try:
    TEST_RECORDS = evaluate_with_diagnostics(
        evaluator, SAMPLE_TASKS, RESULTS_DIR, AGENT_DIR, TASK_TYPES,
        live_observer=LIVE_AGENT_OBSERVER,
    )
finally:
    DIAGNOSTICS_ZIP = archive_diagnostics(RESULTS_DIR)
    print('Download complete diagnostics:', DIAGNOSTICS_ZIP)

# Preserve the starter's downstream predictions variable.
predictions = [{'id': row['instance_id'], 'prediction': row['agent_patch'] or 'NO_PATCH'} for row in TEST_RECORDS]
columns = ['instance_id', 'resolved', 'patch_chars', 'duration_seconds',
           'total_tokens_observed_sum', 'requests_without_usage', 'pending_tool_calls', 'stop_reason']
display(pd.DataFrame(TEST_RECORDS).reindex(columns=columns))
print('Reports:', RESULTS_DIR / 'report')
print('Agent usage and stages:', RESULTS_DIR / 'report' / 'agent_stage_usage.csv')
print('Interaction details:', RESULTS_DIR / 'report' / '<task_id>_interactions.csv')


/usr/local/lib/python3.12/dist-packages/google/adk/features/_feature_decorator.py:72: UserWarning: [EXPERIMENTAL] feature FeatureName.AGENT_CONFIG is enabled.
  check_feature_enabled()
/tmp/ipykernel_163/3247182378.py:97: UserWarning: [EXPERIMENTAL] EventsCompactionConfig: This feature is experimental and may change or be removed in future versions without notice. It may introduce breaking changes at any time.
  events_compaction_config=EventsCompactionConfig(**COMPACTION_SETTINGS),


Effective evaluation limits: {'max_time_minutes': None, 'max_tool_calls': 40, 'max_turns': 80, 'max_total_tokens': None, 'max_cost_usd': None, 'timeout_seconds': 60, 'verification_timeout_seconds': 300, 'max_stdout_chars': 5000, 'max_file_lines': 150, 'max_file_chars': 10000}
Declared models: ['gemma-4-e2b-it-qat-w4a16-ct'] helpers: ['code_analyzer_agent']
Artifacts: /kaggle/working/diagnostic_runs/submission-6-zipnaum33gf
Per-task session time limit: DISABLED. Other limits above still apply.
[candidate 1/30] Evaluating fastapi_15661 (fastapi/fastapi)


INFO:google_adk.google.adk.plugins.plugin_manager:Plugin 'swegemma_display' registered.
INFO:google_adk.google.adk.plugins.plugin_manager:Plugin 'model_retry' registered.
INFO:google_adk.google.adk.plugins.plugin_manager:Plugin 'swegemma_tool_error' registered.


[   0.0s] 📋 System Prompt: Implement the supplied issue in /workspace. Deliver the smallest complete 
implementation patch: cover every requested be...
  [   0.0s] 🎯 Task Prompt: You are evaluating a software engineering task for repository fastapi/fastapi.
  [  10.3s] 💭 (2500 chars) "The user wants me to implement an issue described in the provided text, which seems to
be a template for a Pull Request ..."
  [  10.3s] 📖 read_file (scripts/add_latest_release_date.py)
  [  10.4s]   ✓ {'status': 'ok', 'filepath': 'scripts/add_latest_release_date.py', 'content': '"""Check 
release-notes.md and add today\'…
  [  16.4s] 💭 (2996 chars) "The script `scripts/add_latest_release_date.py` is a Python script that reads 
`docs/en/docs/release-notes.md`, finds rel..."
  [  16.4s] 🤖 submit_patch
  [  16.4s]   → submit_patch: {"status": "ok", "patch_size": 0, "files_changed": 0}
  [  17.1s] ✅ Agent finished: I have analyzed the provided issue and found a relevant script, 
`scripts/add_latest_release_date.py`, which appears to be the implementation for automating release preparation by 
adding the current da...

╭───────────────────────────────────────────────── Latest Detail ─────────────────────────────────────────────────╮
│   ✅ Agent Finished                                                                                             │
│                                                                                                                 │
│   Message:                                                                                                      │
│     I have analyzed the provided issue and found a relevant script, `scripts/add_latest_release_date.py`, which │
│ appears to be the implementation for automating release preparation by adding the current date to release       │
│ headers in `docs/en/docs/release-notes.md`. Since no specific bug or feature request was detailed beyond the    │
│ boilerplate, and the script seems functionally complete for its stated purpose, I have submitted the existing   │
│ implementation.                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯
╭─────────────────────────────────────────── SWE-gemma Agent Evaluator ───────────────────────────────────────────╮
│   [1/30] fastapi_15661 (fastapi/fastapi)  ·  0m 17s                                                             │
│                                                                                                                 │
│           Tools ╸━━━━━━━━━━━━━━━━━━━━━   1 / 40  (39 left)                                                      │
│            Cost ━━━━━━━━━━━━━━━━━━━━━━   $0.00 (Unlimited)                                                      │
│            Time ━━━━━━━━━━━━━━━━━━━━━━   0.3m (Unlimited)                                                       │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

  -> resolved=False, patch_chars=0, tools=1, duration=25.258989616999997, input=14361, output=1568, stop=not recorded
[candidate 2/30] Evaluating fastapi_15588 (fastapi/fastapi)


...

INFO:google_adk.google.adk.plugins.plugin_manager:Plugin 'swegemma_display' registered.
INFO:google_adk.google.adk.plugins.plugin_manager:Plugin 'model_retry' registered.
INFO:google_adk.google.adk.plugins.plugin_manager:Plugin 'swegemma_tool_error' registered.


[   0.0s] 📋 System Prompt: Implement the supplied issue in /workspace. Deliver the smallest complete 
implementation patch: cover every requested be...
  [   0.0s] 🎯 Task Prompt: You are evaluating a software engineering task for repository fastapi/fastapi.
  [   1.5s] 💭 (589 chars) "The user wants to fix an issue in the `fastapi` repository where whitespace in the 
`Authorization` header credentials is..."
  [   1.5s] 📖 read_file (fastapi/security/utils.py)
  [   1.5s]   ✓ {'status': 'ok', 'filepath': 'fastapi/security/utils.py', 'content': 'from typing import 
Optional\n\n\ndef get_authoriza…
  [   5.4s] 💭 (1438 chars) "The current implementation of `get_authorization_scheme_param` is:
```python
def get_authorization_scheme_param(
    aut..."
  [   5.4s] ✏️ edit_file (fastapi/security/utils.py)
  [   5.4s]   ✓ {'status': 'ok', 'filepath': 'fastapi/security/utils.py', 'occurrences': 1, 'strategy': 'regex', 
'diff': '--- a/fastapi/…
  [   6.6s] 💭 (706 chars) "The implementation has been changed to strip whitespace from `param`.

The issue description also mentioned: "Added test..."
  [   6.6s] 🔧 run_command ls ./tests
  [   6.7s]   → benchmarks
  [   8.4s] 💭 (833 chars) "The test output shows a large number of tests, including 
`test_security_http_bearer.py`, which is relevant to the issue...."
  [   8.4s] 🤖 submit_patch
  [   8.5s]   → submit_patch: {"status": "ok", "patch_size": 510, "files_changed": 1}
  [   9.1s] 💭 (154 chars) "The patch was submitted. I have implemented the required change in 
`fastapi/security/utils.py` and verified the process...."
  [   9.1s] ✅ Agent finished: I have implemented the fix by modifying `fastapi/security/utils.py` to apply 
`.strip()` to the extracted parameter, as suggested by the problem description. I have also submitted the patch.

╭───────────────────────────────────────────────── Latest Detail ─────────────────────────────────────────────────╮
│   ✅ Agent Finished                                                                                             │
│                                                                                                                 │
│   Message:                                                                                                      │
│     I have implemented the fix by modifying `fastapi/security/utils.py` to apply `.strip()` to the extracted    │
│ parameter, as suggested by the problem description. I have also submitted the patch.                            │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯
╭─────────────────────────────────────────── SWE-gemma Agent Evaluator ───────────────────────────────────────────╮
│   [30/30] fastapi_14786 (fastapi/fastapi)  ·  0m 09s                                                            │
│                                                                                                                 │
│           Tools ━╸━━━━━━━━━━━━━━━━━━━━   3 / 40  (37 left)                                                      │
│            Cost ━━━━━━━━━━━━━━━━━━━━━━   $0.00 (Unlimited)                                                      │
│            Time ━━━━━━━━━━━━━━━━━━━━━━   0.2m (Unlimited)                                                       │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

  -> resolved=True, patch_chars=510, tools=3, duration=16.655155596999975, input=28597, output=1176, stop=not recorded
Download complete diagnostics: /kaggle/working/diagnostic_runs/submission-6-zipnaum33gf.zip


,instance_id,resolved,patch_chars,duration_seconds,total_tokens_observed_sum,requests_without_usage,pending_tool_calls,stop_reason
0,fastapi_15661,False,0,25.258990,15929,0,0,None
1,fastapi_15588,False,571,38.489480,66336,0,0,None
2,fastapi_15589,False,773,66.511254,146383,0,0,None
3,fastapi_15030,False,5341,85.085227,61159,0,0,None
4,fastapi_14962,False,3209,48.264419,75706,0,0,None
5,fastapi_15280,False,1245,84.672913,102872,0,0,None
6,fastapi_14964,False,1305,28.653643,54941,0,0,None
7,fastapi_15023,False,0,66.218826,104880,0,0,None
8,fastapi_14986,False,9055,206.750086,189166,0,0,None
9,fastapi_14978,False,3035,104.021423,154461,0,0,None


Reports: /kaggle/working/diagnostic_runs/submission-6-zipnaum33gf/report
Agent usage and stages: /kaggle/working/diagnostic_runs/submission-6-zipnaum33gf/report/agent_stage_usage.csv
Interaction details: /kaggle/working/diagnostic_runs/submission-6-zipnaum33gf/report/<task_id>_interactions.csv


In [ ]:
# Create results.zip to downlaod
#zip_base = WORKING_DIR / 'results'
#zip_base = WORKING_DIR / "diagnostic_runs/20261015T210314.380808Z_qgou9ey9"
#zip_path = Path(shutil.make_archive(str(zip_base), 'zip', root_dir="/kaggle/working/diagnostic_runs/20261015T210314.380808Z_qgou9ey9"))

In [9]:
print("done")

done
